# Lab 3 – Clean the NYC Taxi Data, Then Stop
**Course:** INF4058 Data Engineering
**Dataset:** `yellow_tripdata_2021-01.csv.gz` (NYC Yellow Taxi, January 2021)

**Goal:** look at the data, profile it in chunks, find its problems, decide what to do with each one,
and write `clean(df)`. **We stop at cleaning – nothing is loaded into PostgreSQL.**

**Notebook map**
0. Setup
1. Look (first 100 rows)
2. Profile (whole file, chunks of 100,000)
3. Find problems (counts)
4. Decide (policy table)
5. Clean (`clean(df)` + sample test + full run)
6. Verification

## 0. Setup
**Purpose:** import libraries, define constants, and make sure the data file exists.

**Important lines**
- `CHUNK_SIZE = 100_000` – the assignment says to read the file in chunks of exactly 100,000 rows.
- `JAN_START` / `JAN_END` – January boundaries: start **inclusive**, end **exclusive**.
- `DTYPES` – same idea as `ingest_data.py`: nullable `Int64` so integer columns that contain missing values stay integers (a normal `int64` cannot hold missing values and pandas would turn them into floats).
- If the file is missing it is downloaded from the public DataTalksClub mirror of the NYC TLC data (same January 2021 file used in class). Nothing is substituted.

**Expected output:** one line saying the file exists and its size.

In [1]:
import os
import urllib.request
from collections import Counter

import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)

DATA_PATH = os.environ.get("TAXI_CSV", "yellow_tripdata_2021-01.csv.gz")
DATA_URL = ("https://github.com/DataTalksClub/nyc-tlc-data/releases/download/"
            "yellow/yellow_tripdata_2021-01.csv.gz")
CHUNK_SIZE = 100_000

JAN_START = pd.Timestamp("2021-01-01 00:00:00")   # inclusive
JAN_END = pd.Timestamp("2021-02-01 00:00:00")     # exclusive

PU, DO = "tpep_pickup_datetime", "tpep_dropoff_datetime"
DATE_COLS = [PU, DO]
DATE_FORMAT = "%Y-%m-%d %H:%M:%S"

DTYPES = {
    "VendorID": "Int64",
    "passenger_count": "Int64",
    "trip_distance": "float64",
    "RatecodeID": "Int64",
    "store_and_fwd_flag": "string",
    "PULocationID": "Int64",
    "DOLocationID": "Int64",
    "payment_type": "Int64",
    "fare_amount": "float64",
    "extra": "float64",
    "mta_tax": "float64",
    "tip_amount": "float64",
    "tolls_amount": "float64",
    "improvement_surcharge": "float64",
    "total_amount": "float64",
    "congestion_surcharge": "float64",
}

if not os.path.exists(DATA_PATH):
    print("Downloading dataset ...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
assert os.path.exists(DATA_PATH), "dataset not found"
print(f"Dataset OK: {DATA_PATH} ({os.path.getsize(DATA_PATH):,} bytes)")

Dataset OK: yellow_tripdata_2021-01.csv.gz (25,031,880 bytes)


**Helper functions used by every later section**

- `parse_dates(df)` converts the two timestamp columns with `errors="coerce"`: a value that cannot be parsed becomes `NaT` (missing) instead of crashing. This is how *invalid or unparseable pickup timestamps* are handled consistently everywhere (they are counted explicitly later).
- `read_chunks()` yields DataFrames of 100,000 rows, so the whole file is never in memory at once.
- `duplicate_mask()` finds duplicate rows **across chunks** (explained in section 3).

In [2]:
def parse_dates(df):
    """Convert timestamp columns; unparseable values become NaT (never raises)."""
    for c in DATE_COLS:
        df[c] = pd.to_datetime(df[c], format=DATE_FORMAT, errors="coerce")
    return df


def read_chunks(path=DATA_PATH, nrows=None):
    """Yield the file as DataFrames of CHUNK_SIZE rows (dates parsed, dtypes set)."""
    reader = pd.read_csv(path, dtype=DTYPES, chunksize=CHUNK_SIZE, nrows=nrows)
    for chunk in reader:
        yield parse_dates(chunk)


def read_sample(n=5000, path=DATA_PATH):
    """Small sample for testing."""
    return parse_dates(pd.read_csv(path, dtype=DTYPES, nrows=n))


def row_keys(df):
    """128-bit fingerprint per row: two independent 64-bit hashes of all columns."""
    h1 = pd.util.hash_pandas_object(df, index=False).to_numpy()
    h2 = pd.util.hash_pandas_object(df, index=False, hash_key="abcdefghijklmnop").to_numpy()
    return [(int(a) << 64) | int(b) for a, b in zip(h1, h2)]


def duplicate_mask(df, seen):
    """True for every row whose fingerprint was already seen.

    `seen` is a set that is shared between chunks, so a row in chunk 7 that
    equals a row in chunk 2 is caught. The first occurrence is kept.
    """
    mask = []
    for key in row_keys(df):
        if key in seen:
            mask.append(True)
        else:
            seen.add(key)
            mask.append(False)
    return pd.Series(mask, index=df.index)


def as_bool(s):
    """Missing comparison results (from nullable ints) count as False."""
    return s.fillna(False).astype(bool)

## 1. Look – the first 100 rows
**Purpose:** get a feeling for the data before computing anything. We read only 100 rows, with **no** `dtype` and **no** date parsing, so we see what pandas guesses by default.

**Important lines**
- `nrows=100` – read only the first 100 rows.
- `df.head()`, `df.dtypes`, `df.columns`, `df.shape` – the evidence the assignment asks for.
- `df.isna().sum()` – which columns look empty in this small sample.

**Expected output:** a 5-row table, a list of 18 column types, column names, the shape, and a count of missing values per column.

**How to read it**
- *One row = one taxi trip* (one pickup → one dropoff by one vehicle).
- The date columns are `tpep_pickup_datetime` and `tpep_dropoff_datetime`. Notice their dtype is a plain string at this point – pandas does not parse dates unless asked.
- Columns that show missing values in `isna().sum()` "look empty". This sample is only 100 rows, so the full profile in section 2 is what we trust.

In [3]:
df = pd.read_csv(DATA_PATH, nrows=100)

print(df.head())
print()
print(df.dtypes)
print()
print("Columns:", list(df.columns))
print("Shape of the sample:", df.shape)
print()
print("Missing values in the 100-row sample:")
print(df.isna().sum())

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  improvement_surcharge  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1           2.10           1                  N           142            43             2          8.0    3.0      0.5        0.00           0.0                    0.3   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1           0.20           1                  N           238           151             2          3.0    0.5      0.5        0.00           0.0                    0.3   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1          14.70           1                  N           132           165             1         42.0    0.5      0.5        8.65           0.0                    0.3   
3         1  2021-01-01 00:15:48   2

## 2. Profile – the whole file, in chunks
**Purpose:** measure every column (type, missing values, min, max) **without** loading the full file and **without** changing the data.

**How it works**
- For each chunk we add that chunk's null counts to running totals.
- For min/max we take the min/max of the chunk and compare with the best so far (`min(old_min, new_min)`). This gives the true global min/max because min of mins = global min.
- Missing values are skipped (`dropna()`) when computing min/max.
- Datetime columns are compared as timestamps. A timestamp that cannot be parsed becomes `NaT`, so it is counted in `nulls`, not in min/max.

**Expected output:** `print(profile)` – one row per column with `dtype`, `nulls`, `null_pct`, `min`, `max`.

**Limitations of min/max**
1. They ignore missing values.
2. For text columns (`store_and_fwd_flag`) min/max is alphabetical, which has little meaning.
3. Min/max only show the extremes; they do not say how many values are strange (that is section 3).
4. The `dtype` shown is the dtype we *asked* pandas to use (nullable `Int64` etc.), not necessarily what is in the raw CSV.

In [4]:
def profile_file(path=DATA_PATH):
    total = 0
    nulls, mins, maxs, dtypes = Counter(), {}, {}, {}
    for chunk in read_chunks(path):
        total += len(chunk)
        for col in chunk.columns:
            dtypes[col] = str(chunk[col].dtype)
            nulls[col] += int(chunk[col].isna().sum())
            valid = chunk[col].dropna()
            if valid.empty:
                continue
            lo, hi = valid.min(), valid.max()
            mins[col] = lo if col not in mins else min(mins[col], lo)
            maxs[col] = hi if col not in maxs else max(maxs[col], hi)
    cols = list(dtypes)
    prof = pd.DataFrame({
        "dtype": pd.Series(dtypes),
        "nulls": pd.Series(nulls).reindex(cols),
        "null_pct": (pd.Series(nulls).reindex(cols) / total * 100).round(2),
        "min": pd.Series(mins, dtype=object).reindex(cols),
        "max": pd.Series(maxs, dtype=object).reindex(cols),
    })
    return total, prof


TOTAL_ROWS, profile = profile_file()
print(f"Total rows in file: {TOTAL_ROWS:,}")
print(profile)

Total rows in file: 1,369,765
                                dtype  nulls  null_pct                  min                  max
VendorID                        Int64  98352      7.18                    1                    2
tpep_pickup_datetime   datetime64[ns]      0      0.00  2008-12-31 23:05:14  2021-02-22 16:52:16
tpep_dropoff_datetime  datetime64[ns]      0      0.00  2008-12-31 23:07:22  2021-02-22 16:56:15
passenger_count                 Int64  98352      7.18                    0                    8
trip_distance                 float64      0      0.00                  0.0            263163.28
RatecodeID                      Int64  98352      7.18                    1                   99
store_and_fwd_flag             string  98352      7.18                    N                    Y
PULocationID                    Int64      0      0.00                    1                  265
DOLocationID                    Int64      0      0.00                    1                  265


**How to interpret:** look at `nulls`/`null_pct` for columns that are missing; look at `min`/`max` for impossible values (dates outside 2021-01, negative money, huge distances). Those are the leads we follow in section 3.

## 3. Find problems – exact counts
**Purpose:** count how many rows are affected by each problem, over the **entire** file, chunk by chunk.

**Problems checked** (the five required + two extras found during profiling)
1. Pickup outside January 2021 (`>= 2021-01-01` and `< 2021-02-01` is valid)
2. `fare_amount` zero or negative
3. `trip_distance` zero or negative
4. `passenger_count == 0`
5. Duplicate rows
6. *(extra)* dropoff earlier than pickup
7. *(extra)* missing `VendorID` (the profile showed a block of missing fields)

**Missing vs zero/negative:** missing values (`NaN`/`NaT`) are counted **separately** from zero/negative values, because they mean different things (not recorded vs recorded as an odd number).

**Unparseable pickup timestamps:** they become `NaT`, are counted in `pickup_missing_or_unparseable`, and are **not** counted in `pickup_outside_january_2021`.

**Duplicates across chunks:** each row gets a 128-bit fingerprint (two different 64-bit hashes of all columns combined). We store only the fingerprints (not the rows) in a `set` that lives for the whole file, so a duplicate is found even if the two copies are in different chunks, and memory stays small. *Limitation:* it is a hash comparison, so two *different* rows could in theory collide; with 128 bits the chance for ~1.4 million rows is astronomically small (about 1e-26), but it is not literally zero. The first occurrence is kept; every later copy is counted as a duplicate.

**Overlap:** one row can break several rules (e.g. fare 0 *and* distance 0), so the individual counts add up to **more** than the number of distinct bad rows. We print both.

**Expected output:** a table with one row per problem and its count and percent, then the overlap summary.

In [5]:
def count_problems(path=DATA_PATH):
    seen = set()
    total = 0
    c = Counter()
    for chunk in read_chunks(path):
        total += len(chunk)
        pu, do = chunk[PU], chunk[DO]

        pickup_missing = pu.isna()
        pickup_outside = as_bool((pu < JAN_START) | (pu >= JAN_END))
        fare_zero = as_bool(chunk["fare_amount"].eq(0))
        fare_neg = as_bool(chunk["fare_amount"].lt(0))
        dist_zero = as_bool(chunk["trip_distance"].eq(0))
        dist_neg = as_bool(chunk["trip_distance"].lt(0))
        pax_zero = as_bool(chunk["passenger_count"].eq(0))
        drop_before_pick = as_bool(do < pu)
        dup = duplicate_mask(chunk, seen)

        c["pickup_missing_or_unparseable"] += int(pickup_missing.sum())
        c["pickup_outside_january_2021"] += int(pickup_outside.sum())
        c["fare_amount_missing"] += int(chunk["fare_amount"].isna().sum())
        c["fare_amount_zero"] += int(fare_zero.sum())
        c["fare_amount_negative"] += int(fare_neg.sum())
        c["trip_distance_missing"] += int(chunk["trip_distance"].isna().sum())
        c["trip_distance_zero"] += int(dist_zero.sum())
        c["trip_distance_negative"] += int(dist_neg.sum())
        c["passenger_count_zero"] += int(pax_zero.sum())
        c["passenger_count_missing"] += int(chunk["passenger_count"].isna().sum())
        c["duplicate_rows (2nd+ copies)"] += int(dup.sum())
        c["dropoff_before_pickup"] += int(drop_before_pick.sum())
        c["VendorID_missing"] += int(chunk["VendorID"].isna().sum())

        # rows breaking at least one of the rules that the policy DROPS
        any_drop = (pickup_missing | pickup_outside | fare_zero | fare_neg |
                    dist_zero | dist_neg | drop_before_pick | dup)
        c["ROWS with >=1 drop-rule problem (distinct)"] += int(any_drop.sum())
    return total, c


_, counts = count_problems()
problems = pd.DataFrame({"rows": pd.Series(counts)})
problems["pct_of_file"] = (problems["rows"] / TOTAL_ROWS * 100).round(3)
print(problems)

sum_individual = sum(v for k, v in counts.items()
                     if k not in ("passenger_count_zero", "passenger_count_missing",
                                  "VendorID_missing", "fare_amount_missing",
                                  "trip_distance_missing")
                     and not k.startswith("ROWS"))
print()
print("Sum of the individual drop-rule counts :", f"{sum_individual:,}")
print("Distinct rows with >=1 drop-rule problem:", f"{counts['ROWS with >=1 drop-rule problem (distinct)']:,}")
print("Difference (rows counted by more than one rule):",
      f"{sum_individual - counts['ROWS with >=1 drop-rule problem (distinct)']:,}")

                                             rows  pct_of_file
pickup_missing_or_unparseable                   0        0.000
pickup_outside_january_2021                    24        0.002
fare_amount_missing                             0        0.000
fare_amount_zero                              642        0.047
fare_amount_negative                         6769        0.494
trip_distance_missing                           0        0.000
trip_distance_zero                          19952        1.457
trip_distance_negative                          0        0.000
passenger_count_zero                        26726        1.951
passenger_count_missing                     98352        7.180
duplicate_rows (2nd+ copies)                    0        0.000
dropoff_before_pickup                        5642        0.412
VendorID_missing                            98352        7.180
ROWS with >=1 drop-rule problem (distinct)  32027        2.338

Sum of the individual drop-rule counts : 33,029
Distin

**How to interpret:** each row is an *independent* count over the raw file, so they overlap. The last block shows how much: if the sum of individual counts is larger than the distinct count, some rows break more than one rule. In section 5 the rules run one after another and each row is charged to the **first** rule that catches it, so those numbers add up exactly.

## 4. Decide – the cleaning policy
**Purpose:** for each problem choose `drop`, `fix` or `flag`, say why, and say who should own the rule.

> This table is the **policy** (what we decided). Section 5 is the **implementation** (the code that applies it). They are kept separate on purpose.

**Why not just delete everything suspicious?** A row is only dropped when the trip cannot be trusted or cannot be a normal trip. When the trip itself is real and only one field is doubtful (passenger count), we **flag** it so analysts can decide. We never *fix* by inventing a value: there is no reliable source for a "correct" passenger count, fare or pickup time, so no rule uses `fix`.

**Expected output:** a printed table with columns `problem`, `decision`, `reason`, `rule_owner`.

In [6]:
decisions = pd.DataFrame([
    ("pickup timestamp missing/unparseable", "drop",
     "No reliable time, so the trip cannot be placed in January.", "data engineering"),
    ("pickup outside Jan 2021", "drop",
     "Wrong month for this file; timestamps are unreliable (e.g. 2008).", "data engineering"),
    ("dropoff before pickup", "drop",
     "Negative duration is impossible; cannot be repaired reliably.", "data engineering"),
    ("fare_amount <= 0", "drop",
     "Not a normal paid trip (refund/dispute/test); what to do is a business call.", "business / data owner"),
    ("trip_distance <= 0", "drop",
     "No movement: cancelled or test trip; not a real journey.", "business / data owner"),
    ("duplicate rows", "drop",
     "Same trip recorded twice; keep the first copy only.", "data engineering"),
    ("passenger_count == 0", "flag",
     "Driver-entered field; trip is real, so keep it and mark it.", "business / data owner"),
    ("passenger_count missing", "flag",
     "Value unknown; do not invent one. Keep the row and mark it.", "business / data owner"),
    ("VendorID (and related fields) missing", "flag",
     "Trip data is otherwise valid; missing vendor info is a source gap.", "data engineering"),
], columns=["problem", "decision", "reason", "rule_owner"])
print(decisions.to_string(index=False))

                              problem decision                                                                       reason            rule_owner
 pickup timestamp missing/unparseable     drop                   No reliable time, so the trip cannot be placed in January.      data engineering
              pickup outside Jan 2021     drop            Wrong month for this file; timestamps are unreliable (e.g. 2008).      data engineering
                dropoff before pickup     drop                Negative duration is impossible; cannot be repaired reliably.      data engineering
                     fare_amount <= 0     drop Not a normal paid trip (refund/dispute/test); what to do is a business call. business / data owner
                   trip_distance <= 0     drop                     No movement: cancelled or test trip; not a real journey. business / data owner
                       duplicate rows     drop                          Same trip recorded twice; keep the first copy only. 

## 5. Clean – `clean(df)`
**Purpose:** implement the policy from section 4.

**Contract**
- Input: a DataFrame (one chunk or any piece of the data).
- Output: `(cleaned_df, removed)` where `removed` is a dict `{rule: rows_removed}`.
- The input is **never modified** (`df.copy()` first).
- Rows are **never dropped silently**: every drop is counted in `removed`, even if the count is 0.
- `flag` rules keep the row and add a boolean column: `passenger_count_zero_flag`, `passenger_count_missing_flag`, `vendor_missing_flag`.

**Order and no double counting.** Rules run in a fixed order and each rule only looks at the rows that survived the earlier ones. A row that breaks two rules is therefore counted **once**, under the first rule that catches it. Because of this, the per-rule numbers here can be smaller than the independent counts in section 3, but they add up exactly to rows removed.

**Duplicates across chunks.** `clean` accepts an optional `seen_keys` set. When the same set is passed for every chunk, duplicates are removed **globally** (not only inside a chunk). If you call `clean(df)` without it, only duplicates inside that one DataFrame are removed – a documented limitation, demonstrated in section 6.

**Expected output:** first a test on a 5,000-row sample, then the full-file run with rows before/after, removed per rule and flag totals.

In [7]:
RULE_ORDER = [
    "pickup_missing_or_unparseable",
    "pickup_outside_january_2021",
    "dropoff_before_pickup",
    "fare_amount<=0",
    "trip_distance<=0",
    "duplicate_rows",
]
FLAG_COLS = ["passenger_count_zero_flag", "passenger_count_missing_flag", "vendor_missing_flag"]


def clean(df, seen_keys=None):
    """Apply the cleaning policy. Returns (cleaned_df, {rule: rows_removed})."""
    if seen_keys is None:
        seen_keys = set()          # duplicates are then only detected inside df
    out = df.copy()                # never touch the caller's DataFrame
    removed = {}

    def drop(rule, bad):
        nonlocal out
        bad = as_bool(bad)
        removed[rule] = int(bad.sum())
        out = out[~bad]

    drop("pickup_missing_or_unparseable", out[PU].isna())
    drop("pickup_outside_january_2021", (out[PU] < JAN_START) | (out[PU] >= JAN_END))
    drop("dropoff_before_pickup", out[DO] < out[PU])
    drop("fare_amount<=0", out["fare_amount"] <= 0)
    drop("trip_distance<=0", out["trip_distance"] <= 0)
    drop("duplicate_rows", duplicate_mask(out, seen_keys))

    # FLAG rules: keep the rows, add columns
    out = out.assign(
        passenger_count_zero_flag=as_bool(out["passenger_count"].eq(0)),
        passenger_count_missing_flag=out["passenger_count"].isna(),
        vendor_missing_flag=out["VendorID"].isna(),
    )
    return out, removed


# ---- 5a. test on a small sample first ---------------------------------
sample = read_sample(5000)
sample_clean, sample_removed = clean(sample)
print("Sample rows before:", len(sample))
print("Sample rows after :", len(sample_clean))
print("Removed per rule  :", sample_removed)
print("Sample flag counts:", {f: int(sample_clean[f].sum()) for f in FLAG_COLS})

Sample rows before: 5000
Sample rows after : 4775
Removed per rule  : {'pickup_missing_or_unparseable': 0, 'pickup_outside_january_2021': 16, 'dropoff_before_pickup': 0, 'fare_amount<=0': 60, 'trip_distance<=0': 149, 'duplicate_rows': 0}
Sample flag counts: {'passenger_count_zero_flag': 83, 'passenger_count_missing_flag': 0, 'vendor_missing_flag': 0}


In [8]:
# ---- 5b. full file, 100,000 rows at a time ---------------------------------
def clean_file(path=DATA_PATH, nrows=None, check=True):
    """Clean the whole file chunk by chunk. Nothing is written anywhere."""
    seen = set()                       # shared across chunks -> global de-duplication
    before = after = 0
    removed_total = Counter()
    flag_total = Counter()
    for chunk in read_chunks(path, nrows):
        before += len(chunk)
        cleaned, removed = clean(chunk, seen)
        after += len(cleaned)
        removed_total.update(removed)
        for f in FLAG_COLS:
            flag_total[f] += int(cleaned[f].sum())
        if check:                      # every cleaned chunk must satisfy the rules
            assert cleaned[PU].between(JAN_START, JAN_END, inclusive="left").all()
            assert (cleaned["fare_amount"] > 0).all()
            assert (cleaned["trip_distance"] > 0).all()
            assert (cleaned[DO] >= cleaned[PU]).all()
    # `seen` only holds fingerprints of rows that SURVIVED -> if it has exactly
    # `after` entries, every kept row is unique across the whole file.
    return before, after, {r: removed_total[r] for r in RULE_ORDER}, dict(flag_total), len(seen)


ROWS_BEFORE, ROWS_AFTER, REMOVED, FLAGS, N_UNIQUE_KEPT = clean_file()
ROWS_REMOVED = ROWS_BEFORE - ROWS_AFTER

print(f"Rows before cleaning : {ROWS_BEFORE:,}")
print(f"Rows after cleaning  : {ROWS_AFTER:,}")
print(f"Rows removed         : {ROWS_REMOVED:,}")
print()
print("Rows removed per rule ({rule: rows_removed}):")
removed_table = pd.DataFrame({"rows_removed": pd.Series(REMOVED)})
removed_table["pct_of_file"] = (removed_table["rows_removed"] / ROWS_BEFORE * 100).round(3)
print(removed_table)
print()
print("Sum of per-rule removals:", f"{sum(REMOVED.values()):,}")
print()
print("Flagged rows (kept in the data, not removed):")
flag_table = pd.DataFrame({"rows_flagged": pd.Series(FLAGS)})
flag_table["pct_of_kept_rows"] = (flag_table["rows_flagged"] / ROWS_AFTER * 100).round(3)
print(flag_table)

Rows before cleaning : 1,369,765
Rows after cleaning  : 1,337,738
Rows removed         : 32,027

Rows removed per rule ({rule: rows_removed}):
                               rows_removed  pct_of_file
pickup_missing_or_unparseable             0        0.000
pickup_outside_january_2021              24        0.002
dropoff_before_pickup                  5642        0.412
fare_amount<=0                         7394        0.540
trip_distance<=0                      18967        1.385
duplicate_rows                            0        0.000

Sum of per-rule removals: 32,027

Flagged rows (kept in the data, not removed):
                              rows_flagged  pct_of_kept_rows
passenger_count_zero_flag            26246             1.962
passenger_count_missing_flag         91389             6.832
vendor_missing_flag                  91389             6.832


**How to interpret**
- `rows before - rows after` must equal the sum of the per-rule removals (checked again in Verification).
- A rule's number here can be **lower** than its independent count in section 3 because earlier rules already removed some of the same rows. That is the overlap, not an error.
- Flagged rows are still inside `rows after`; the flag columns tell downstream users which rows to treat carefully.
- The cleaned data is **not** written to disk or loaded into Postgres – we stop here, as required.

## 6. Verification
**Purpose:** prove (with assertions, not with words) that the cleaning does what the policy says.

Each test is wrapped in `check(name, condition)`: it raises an `AssertionError` if the condition is false and otherwise records `PASS`. A test is only shown as PASS if it really ran and held.

**Tests**
1. `clean` returns a `(DataFrame, dict)` pair with integer counts.
2. The input DataFrame is unchanged after cleaning.
3. A hand-made test frame with known problems gives the exact expected counts.
4. After the date rule, all pickups are inside `[2021-01-01, 2021-02-01)`.
5. Flagged rows are retained and flagged (not dropped).
6. Duplicates: removed within a frame; across two "chunks" only when `seen_keys` is shared (documented policy).
7. Row-count reconciliation (test frame, sample, and full file).
8. Reproducibility: same input → same output and same counts, twice.
9. Whole-file uniqueness: the number of fingerprints kept equals rows after cleaning.
10. The full-file removal counts equal the counts from a second, independent run.

**Expected output:** a table of test names with `PASS`.

In [9]:
results = []


def check(name, condition):
    assert condition, f"FAILED: {name}"
    results.append((name, "PASS"))


# --- a small hand-made frame whose answer we know in advance ----------------
t = sample.iloc[[0] * 10].reset_index(drop=True)       # 10 copies of one valid row
t.loc[1, PU] = pd.Timestamp("2020-12-31 23:59:59")     # before January
t.loc[2, PU] = pd.Timestamp("2021-02-01 00:00:00")     # exactly the (exclusive) end
t.loc[3, "fare_amount"] = 0.0
t.loc[4, "fare_amount"] = -5.0
t.loc[5, "trip_distance"] = 0.0
t.loc[6, "passenger_count"] = 0                        # flag only
t.loc[7, "passenger_count"] = pd.NA                    # flag only
t.loc[9, PU] = pd.NaT                                  # unparseable/missing pickup
# row 8 stays an exact copy of row 0  -> duplicate
# (row 0 is valid; the other rows each differ from it)

t_before = t.copy(deep=True)
t_clean, t_removed = clean(t)

# 1. return types
res = clean(t)
check("clean() returns a 2-tuple", isinstance(res, tuple) and len(res) == 2)
check("first output is a DataFrame", isinstance(res[0], pd.DataFrame))
check("second output is a dict of ints", isinstance(res[1], dict)
      and all(isinstance(v, int) for v in res[1].values()))
check("dict has an entry for every rule", list(res[1]) == RULE_ORDER)

# 2. input unchanged
pd.testing.assert_frame_equal(t, t_before)
check("input DataFrame unchanged (values, dtypes, columns)", True)
check("no flag columns leaked into the input", not any(f in t.columns for f in FLAG_COLS))

# 3. exact expected counts on the test frame
expected = {
    "pickup_missing_or_unparseable": 1,
    "pickup_outside_january_2021": 2,
    "dropoff_before_pickup": 0,     # row 2 also has dropoff<pickup but the date rule caught it first
    "fare_amount<=0": 2,
    "trip_distance<=0": 1,
    "duplicate_rows": 1,
}
check("test-frame removal counts match expected", t_removed == expected)
check("no double counting (row 2 charged once)", sum(t_removed.values()) == 7)

# 4. date rule
check("all pickups inside [2021-01-01, 2021-02-01)",
      t_clean[PU].between(JAN_START, JAN_END, inclusive="left").all())
check("boundary: 2021-02-01 00:00:00 removed (end is exclusive)",
      pd.Timestamp("2021-02-01 00:00:00") not in set(t_clean[PU]))

# 5. flagged rows are retained
check("flagged rows retained (3 rows left: valid, pax=0, pax=NA)", len(t_clean) == 3)
check("passenger_count == 0 row kept and flagged",
      int(t_clean["passenger_count_zero_flag"].sum()) == 1)
check("passenger_count missing row kept and flagged",
      int(t_clean["passenger_count_missing_flag"].sum()) == 1)
check("flag rules removed nothing", "passenger_count" not in "".join(t_removed))

# 6. duplicate policy
check("duplicate inside one frame removed", t_removed["duplicate_rows"] == 1)
shared = set()
_, r_a = clean(t.iloc[:5], shared)
_, r_b = clean(t.iloc[5:], shared)           # row 8 (copy of row 0) is in this half
check("duplicate ACROSS chunks removed when seen_keys is shared",
      r_a["duplicate_rows"] + r_b["duplicate_rows"] == 1)
_, r_c = clean(t.iloc[:5])
_, r_d = clean(t.iloc[5:])
check("same split WITHOUT shared seen_keys misses it (documented limitation)",
      r_c["duplicate_rows"] + r_d["duplicate_rows"] == 0)

# 7. reconciliation (test frame + 5,000-row sample)
check("test frame: before - after == sum(removed)",
      len(t) - len(t_clean) == sum(t_removed.values()))
check("sample: before - after == sum(removed)",
      len(sample) - len(sample_clean) == sum(sample_removed.values()))
check("full file: before - after == sum(removed per rule)",
      ROWS_BEFORE - ROWS_AFTER == sum(REMOVED.values()))
check("full file: before == total rows profiled", ROWS_BEFORE == TOTAL_ROWS)

# 8. reproducibility
a1, a2 = clean(sample), clean(sample)
pd.testing.assert_frame_equal(a1[0], a2[0])
check("same input twice -> identical DataFrame", True)
check("same input twice -> identical counts", a1[1] == a2[1])
r1 = clean_file(nrows=300_000, check=True)
r2 = clean_file(nrows=300_000, check=True)
check("chunked run twice (first 300,000 rows) -> identical results", r1 == r2)

# 9. global uniqueness
check("full file: kept-row fingerprints == rows after (no duplicate survives)",
      N_UNIQUE_KEPT == ROWS_AFTER)

# 10. second independent full run gives the same numbers
again = clean_file()
check("second full run reproduces rows/removed/flags",
      again[:4] == (ROWS_BEFORE, ROWS_AFTER, REMOVED, FLAGS))

verification = pd.DataFrame(results, columns=["test", "result"])
verification.index = verification.index + 1
print(verification.to_string())
print()
print(f"{sum(r == 'PASS' for _, r in results)} of {len(results)} tests passed.")

                                                                      test result
1                                                clean() returns a 2-tuple   PASS
2                                              first output is a DataFrame   PASS
3                                          second output is a dict of ints   PASS
4                                         dict has an entry for every rule   PASS
5                      input DataFrame unchanged (values, dtypes, columns)   PASS
6                                    no flag columns leaked into the input   PASS
7                                 test-frame removal counts match expected   PASS
8                                  no double counting (row 2 charged once)   PASS
9                              all pickups inside [2021-01-01, 2021-02-01)   PASS
10                boundary: 2021-02-01 00:00:00 removed (end is exclusive)   PASS
11               flagged rows retained (3 rows left: valid, pax=0, pax=NA)   PASS
12              

**How to interpret:** if any test failed, the notebook would stop with an `AssertionError` at that test and the table would not be printed. The table only exists if every test passed.

## Stop here
Cleaning is finished. The cleaned data is **not** written to disk and **not** loaded into PostgreSQL. Next session we connect this `clean()` step to the pipeline.

In [10]:
import json
os.makedirs("results", exist_ok=True)          # results/ is git-ignored
with open("results/summary.json", "w") as f:
    json.dump({
        "total_rows": TOTAL_ROWS,
        "rows_before": ROWS_BEFORE, "rows_after": ROWS_AFTER, "rows_removed": ROWS_REMOVED,
        "removed": REMOVED, "flags": FLAGS,
        "problems": {k: int(v) for k, v in counts.items()},
        "tests_passed": len(results),
    }, f, indent=2)
print("summary saved to results/summary.json (git-ignored)")

summary saved to results/summary.json (git-ignored)
